In [0]:
# Databricks notebook source
# =============================================================
# Unit tests — bronze catalogs_raw  (plain notebook, no pytest)
# =============================================================

import sys
import types

# Mock the decorator so the import doesn't crash
mock_pipelines = types.ModuleType("pipelines")
mock_pipelines.table = lambda **kwargs: (lambda fn: fn)  # no-op decorator
sys.modules["pyspark.pipelines"] = mock_pipelines

from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType,
)

# -------------------------------------------------------------
# Copy of COLUMN_MAPPING from bronze_catalogs_raw.py
# -------------------------------------------------------------
COLUMN_MAPPING = {
    "Время разгона 0-100 км/ч, с": "acceleration_0_100",
    "Клиренс":                     "clearance",
    "Кол-во мест":                 "seats",
    "Комплектация":                "complectation",
    "Коробка передач":             "gearbox",
    "Максимальная скорость, км/ч": "max_speed",
    "Марка":                       "marka",
    "Марка кузова":                "body_mark",
    "Модель":                      "model",
    "Мощность двигателя":          "power",
    "Объем багажника":             "trunk_volume",
    "Объём двигателя":             "engine_volume",
    "Период выпуска":              "production_period",
    "Поколение":                   "generation",
    "Привод":                      "drive",
    "Расход топлива":              "fuel_consumption",
    "Страна сборки":               "country",
    "Тип кузова":                  "body_type",
    "Тип топлива":                 "fuel_type",
}

# -------------------------------------------------------------
# Copy of add_lineage from bronze_catalogs_raw.py
# -------------------------------------------------------------
def add_lineage(df):
    # Rename Russian → English
    for russian, english in COLUMN_MAPPING.items():
        df = df.withColumnRenamed(russian, english)

    # Add lineage from _metadata
    return df.select(
        "*",
        col("_metadata.file_path").alias("source_file_path"),
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_modification_time").alias("source_file_modified_at"),
    )


# -------------------------------------------------------------
# Helper — build the test DataFrame once
# Simulates what Auto Loader produces after reading catalogs.json
# -------------------------------------------------------------
def _catalogs_df():
    schema = StructType([
        StructField("loaded_at",                        StringType(),    True),
        StructField("source_file",                      StringType(),    True),
        StructField("Время разгона 0-100 км/ч, с",     StringType(),    True),
        StructField("Клиренс",                          StringType(),    True),
        StructField("Кол-во мест",                      StringType(),    True),
        StructField("Комплектация",                     StringType(),    True),
        StructField("Коробка передач",                  StringType(),    True),
        StructField("Максимальная скорость, км/ч",     StringType(),    True),
        StructField("Марка",                            StringType(),    True),
        StructField("Марка кузова",                     StringType(),    True),
        StructField("Модель",                           StringType(),    True),
        StructField("Мощность двигателя",              StringType(),    True),
        StructField("Объем багажника",                  StringType(),    True),
        StructField("Объём двигателя",                  StringType(),    True),
        StructField("Период выпуска",                   StringType(),    True),
        StructField("Поколение",                        StringType(),    True),
        StructField("Привод",                           StringType(),    True),
        StructField("Расход топлива",                   StringType(),    True),
        StructField("Страна сборки",                    StringType(),    True),
        StructField("Тип кузова",                       StringType(),    True),
        StructField("Тип топлива",                      StringType(),    True),
        StructField("_rescued_data",                    StringType(),    True),
        StructField("_corrupt_record",                  StringType(),    True),
        StructField("_metadata", StructType([
            StructField("file_path",              StringType(),    True),
            StructField("file_name",              StringType(),    True),
            StructField("file_modification_time", TimestampType(), True),
        ]), True),
    ])

    data = [
        # valid row
        Row("2026-02-02T14:02:02.236Z", "/path/catalogs.csv",
            "7.7", "158 мм", "5 мест", "3.2 AT CL", "АКПП", "245",
            "Acura", "YA4", "CL", "225 л.с.", "385 л", "3.2 л",
            "2002 -2003", "2 поколение", "Передний (FF)", "10,2 л",
            "США", "Купе", "Бензин АИ-92",
            None, None,                                                          # no rescued, no corrupt
            Row("/path/catalogs.json", "catalogs.json", None)),

        # rescued row — extra column in JSON
        Row("2026-02-02T14:02:02.236Z", "/path/catalogs.csv",
            "6.3", "158 мм", "5 мест", "3.2 MT Type S", "МКПП", "250",
            "Acura", "YA4", "CL", "260 л.с.", "385 л", "3.2 л",
            "2002 -2003", "2 поколение", "Передний (FF)", "10,7 л",
            "США", "Купе", "Бензин АИ-92",
            '{"extra_col":"some_value"}', None,                                  # rescued
            Row("/path/catalogs.json", "catalogs.json", None)),

        # corrupt row — malformed JSON line
        Row(None, None,
            None, None, None, None, None, None,
            None, None, None, None, None, None,
            None, None, None, None,
            None, None, None,
            None, '{"malformed json line"}',                                     # corrupt
            Row("/path/catalogs.json", "catalogs.json", None)),
    ]

    return spark.createDataFrame(data, schema)


# -------------------------------------------------------------
# Tests
# -------------------------------------------------------------

# --- Rename tests ---
def test_russian_columns_renamed():
    out = add_lineage(_catalogs_df())
    for english in COLUMN_MAPPING.values():
        assert english in out.columns, f"Column '{english}' missing after rename"
    print("PASSED  test_russian_columns_renamed")


def test_russian_columns_gone():
    out = add_lineage(_catalogs_df())
    for russian in COLUMN_MAPPING.keys():
        assert russian not in out.columns, f"Russian column '{russian}' still exists after rename"
    print("PASSED  test_russian_columns_gone")


# --- Lineage tests ---
def test_lineage_columns_exist():
    out = add_lineage(_catalogs_df())
    cols = out.columns
    assert "source_file_path"        in cols
    assert "source_file_name"        in cols
    assert "source_file_modified_at" in cols
    print("PASSED  test_lineage_columns_exist")


# --- Rescued / corrupt tests ---
def test_rescued_rows_preserved():
    out = add_lineage(_catalogs_df())
    assert out.filter(col("_rescued_data").isNotNull()).count() == 1
    print("PASSED  test_rescued_rows_preserved")


def test_corrupt_rows_preserved():
    out = add_lineage(_catalogs_df())
    assert out.filter(col("_corrupt_record").isNotNull()).count() == 1
    print("PASSED  test_corrupt_rows_preserved")


def test_row_not_both_corrupt_and_rescued():
    out = add_lineage(_catalogs_df())
    bad = out.filter(
        col("_corrupt_record").isNotNull() &
        col("_rescued_data").isNotNull()
    ).count()
    assert bad == 0
    print("PASSED  test_row_not_both_corrupt_and_rescued")


# --- Row count ---
def test_row_count_preserved():
    out = add_lineage(_catalogs_df())
    assert out.count() == 3
    print("PASSED  test_row_count_preserved")


# -------------------------------------------------------------
# Run all
# -------------------------------------------------------------
tests = [
    # rename
    test_russian_columns_renamed,
    test_russian_columns_gone,

    # lineage
    test_lineage_columns_exist,

    # rescued / corrupt
    test_rescued_rows_preserved,
    test_corrupt_rows_preserved,
    test_row_not_both_corrupt_and_rescued,

    # row count
    test_row_count_preserved,
]

passed, failed = 0, 0

for t in tests:
    try:
        t()
        passed += 1
    except Exception as e:
        print(f"FAILED  {t.__name__} — {e}")
        failed += 1

print(f"\n{passed} passed, {failed} failed out of {len(tests)} tests")